# Módulo 9 · Clase 8 — Despliegue Local

**Proyecto Final: Sistema Inteligente de Predicción y Exposición de Modelos**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Producción vs desarrollo |
| 2 | Servicio `systemd` en Linux |
| 3 | Docker: Dockerfile + imagen |
| 4 | Docker Compose — API + base de datos |
| 5 | Checklist de producción |

---
## 1. Producción vs desarrollo

| Aspecto | Desarrollo | Producción |
|---------|------------|------------|
| **Servidor** | `uvicorn --reload` | `gunicorn + uvicorn workers` |
| **Debug** | logs detallados | solo errores |
| **Secretos** | `.env` local | variables de entorno del sistema |
| **Modelos** | archivo local | volumen Docker / S3 |
| **BD** | SQLite | PostgreSQL / cloud |
| **Restart** | manual | automático (systemd / Docker) |
| **HTTPS** | no | nginx + certificado |

En esta clase cubrimos el **despliegue local** (servidor Linux o máquina propia), que es el paso previo al despliegue en la nube.

In [ ]:
from pathlib import Path
import subprocess, json

# Comprobar que el entorno está listo
for p in ['api/main.py', 'models/rf_vinos_v1.joblib', 'requirements.txt']:
    existe = Path(p).exists()
    estado = '✓' if existe else '✗ FALTA'
    print(f"  {estado} {p}")

print("\nSi falta algún archivo, ejecuta primero las clases anteriores.")

---
## 2. Servicio `systemd` en Linux

`systemd` es el sistema de inicio de Linux. Permite que un proceso arranque automáticamente con el sistema, se reinicie si falla y registre sus logs.

In [ ]:
# Generar el archivo de unidad systemd
# NOTA: Para instalar de verdad necesitas permisos root en Linux
# Este bloque genera el archivo para que puedas revisarlo

import getpass, os

PROYECTO_DIR = Path.cwd().absolute()
VENV_DIR     = PROYECTO_DIR / '.venv'
USUARIO      = getpass.getuser()

systemd_unit = f"""[Unit]
Description=Wine Quality API
After=network.target

[Service]
Type=simple
User={USUARIO}
WorkingDirectory={PROYECTO_DIR}
Environment=PYTHONPATH={PROYECTO_DIR}
ExecStart={VENV_DIR}/bin/gunicorn api.main:app \\
    --workers 2 \\
    --worker-class uvicorn.workers.UvicornWorker \\
    --bind 0.0.0.0:8000 \\
    --log-level warning \\
    --access-logfile logs/access.log \\
    --error-logfile logs/error.log
Restart=always
RestartSec=5

[Install]
WantedBy=multi-user.target
"""

Path('docs/wine-api.service').write_text(systemd_unit)
print(systemd_unit)
print("\nGuardado en docs/wine-api.service")

In [ ]:
print("Comandos para instalar el servicio (requiere root en Linux):")
print("""
# Copiar la unidad al directorio de systemd
sudo cp docs/wine-api.service /etc/systemd/system/

# Recargar systemd para que reconozca la nueva unidad
sudo systemctl daemon-reload

# Habilitar el servicio (arranca en cada reinicio del sistema)
sudo systemctl enable wine-api

# Arrancar ahora
sudo systemctl start wine-api

# Verificar estado
sudo systemctl status wine-api

# Ver logs en tiempo real
sudo journalctl -u wine-api -f

# Parar / reiniciar
sudo systemctl stop wine-api
sudo systemctl restart wine-api
"""
)

### ¿Por qué `gunicorn` en lugar de `uvicorn` solo?

- `uvicorn` es el servidor ASGI (asíncrono). Excelente, pero si el proceso muere, la API desaparece.
- `gunicorn` es el **process manager**: gestiona múltiples workers y los reinicia si fallan.
- Combinación estándar: `gunicorn --worker-class uvicorn.workers.UvicornWorker`

In [ ]:
# Prueba de gunicorn localmente
import subprocess, time, requests

proc = subprocess.Popen(
    ['python', '-m', 'gunicorn', 'api.main:app',
     '--workers', '2',
     '--worker-class', 'uvicorn.workers.UvicornWorker',
     '--bind', '0.0.0.0:8002',
     '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)

if proc.poll() is None:
    r = requests.get('http://localhost:8002/health')
    print(f"✓ gunicorn arrancado — health: {r.json()}")
    print(f"  2 workers activos")
    proc.terminate(); proc.wait()
    print("✓ Servidor detenido")
else:
    stderr = proc.stderr.read().decode()
    print(f"Error (¿gunicorn instalado? pip install gunicorn):\n{stderr[:300]}")

---
## 3. Docker: Dockerfile + imagen

In [ ]:
%%writefile Dockerfile
# Base: Python 3.11 slim (imagen pequeña, sin GUI)
FROM python:3.11-slim

# Evitar interacción en instalaciones
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

# Directorio de trabajo dentro del contenedor
WORKDIR /app

# Copiar solo requirements primero (aprovecha la caché de capas de Docker)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiar el código fuente
COPY api/ ./api/
COPY models/ ./models/

# Crear directorios de datos en runtime
RUN mkdir -p data/raw data/processed data/db logs

# Exponer el puerto
EXPOSE 8000

# Comando de inicio en producción
CMD ["gunicorn", "api.main:app",
     "--workers", "2",
     "--worker-class", "uvicorn.workers.UvicornWorker",
     "--bind", "0.0.0.0:8000",
     "--log-level", "warning"]

In [ ]:
print("Comandos Docker básicos:")
print("""
# Construir la imagen (etiquetada como 'wine-api')
docker build -t wine-api .

# Ejecutar el contenedor
docker run -d \\
    --name wine-api-container \\
    -p 8000:8000 \\
    wine-api

# Ver logs
docker logs wine-api-container -f

# Entrar en el contenedor
docker exec -it wine-api-container bash

# Parar y eliminar
docker stop wine-api-container
docker rm wine-api-container

# Listar imágenes
docker images
""")

In [ ]:
# Verificar si Docker está disponible en el sistema
result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✓ Docker disponible: {result.stdout.strip()}")

    # Construir la imagen si Docker está instalado
    print("\nConstruyendo imagen Docker... (puede tardar 2-5 minutos la primera vez)")
    build = subprocess.run(
        ['docker', 'build', '-t', 'wine-api:v1', '.'],
        capture_output=True, text=True
    )
    if build.returncode == 0:
        print("✓ Imagen construida: wine-api:v1")
        # Tamaño de la imagen
        size = subprocess.run(['docker', 'image', 'inspect', 'wine-api:v1',
                               '--format', '{{.Size}}'],
                              capture_output=True, text=True)
        mb = int(size.stdout.strip()) / 1024**2
        print(f"  Tamaño: {mb:.0f} MB")
    else:
        print(f"Error en el build: {build.stderr[-500:]}")
else:
    print("Docker no está instalado en este sistema.")
    print("  Para instalarlo: https://docs.docker.com/get-docker/")

---
## 4. Docker Compose — múltiples servicios

In [ ]:
%%writefile docker-compose.yml
version: '3.9'

services:

  api:
    build: .
    image: wine-api:latest
    container_name: wine_api
    ports:
      - "8000:8000"
    volumes:
      # Persistir los datos y modelos fuera del contenedor
      - ./data:/app/data
      - ./models:/app/models
      - ./logs:/app/logs
    environment:
      - PYTHONUNBUFFERED=1
      - LOG_LEVEL=warning
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 20s

  # Nginx como proxy inverso (maneja HTTPS y balanceo)
  nginx:
    image: nginx:alpine
    container_name: wine_nginx
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/conf.d/default.conf:ro
    depends_on:
      api:
        condition: service_healthy
    restart: unless-stopped

volumes:
  data:
  models:

In [ ]:
%%writefile nginx.conf
upstream wine_api {
    server api:8000;
}

server {
    listen 80;
    server_name localhost;

    # Limitar tamaño de petición
    client_max_body_size 1M;

    location / {
        proxy_pass         http://wine_api;
        proxy_set_header   Host $host;
        proxy_set_header   X-Real-IP $remote_addr;
        proxy_set_header   X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_read_timeout 60s;
    }

    # Endpoint de health check directo (sin pasar por FastAPI)
    location /nginx-health {
        return 200 'ok';
        add_header Content-Type text/plain;
    }
}

In [ ]:
print("Comandos Docker Compose:")
print("""
# Arrancar todos los servicios en segundo plano
docker compose up -d

# Ver estado de los servicios
docker compose ps

# Ver logs de todos los servicios
docker compose logs -f

# Ver logs solo de la API
docker compose logs api -f

# Parar sin eliminar los contenedores
docker compose stop

# Parar y eliminar contenedores (datos en volúmenes persisten)
docker compose down

# Reconstruir la imagen y reiniciar
docker compose up -d --build

# Entrar en el contenedor de la API
docker compose exec api bash
""")

---
## 5. Checklist de producción

In [ ]:
import requests, json, time
from pathlib import Path

def checklist_produccion(base_url: str = 'http://localhost:8000') -> dict:
    """Verifica los requisitos mínimos de un despliegue en producción."""
    resultados = {}

    # 1. Health check
    try:
        r = requests.get(f'{base_url}/health', timeout=5)
        resultados['health_check'] = r.status_code == 200
    except Exception as e:
        resultados['health_check'] = False

    # 2. Documentación accesible
    try:
        r = requests.get(f'{base_url}/docs', timeout=5)
        resultados['docs_accesible'] = r.status_code == 200
    except Exception:
        resultados['docs_accesible'] = False

    # 3. Predicción funciona
    vino_test = {
        "tipo": "blanco", "fixed_acidity": 6.5, "volatile_acidity": 0.35,
        "citric_acid": 0.3, "residual_sugar": 7.0, "chlorides": 0.04,
        "free_sulfur_dioxide": 25.0, "total_sulfur_dioxide": 120.0,
        "density": 0.9942, "pH": 3.3, "sulphates": 0.55, "alcohol": 11.5
    }
    try:
        r = requests.post(f'{base_url}/predict', json=vino_test, timeout=10)
        res = r.json()
        resultados['prediccion_ok'] = (
            r.status_code == 200 and
            'prediccion' in res and
            'probabilidad' in res
        )
    except Exception:
        resultados['prediccion_ok'] = False

    # 4. Validación de errores funciona (422)
    try:
        r = requests.post(f'{base_url}/predict',
                          json={**vino_test, 'alcohol': 999}, timeout=5)
        resultados['validacion_errores'] = r.status_code == 422
    except Exception:
        resultados['validacion_errores'] = False

    # 5. Archivos críticos presentes
    archivos_criticos = [
        'api/main.py', 'models/rf_vinos_v1.joblib',
        'requirements.txt', 'Dockerfile'
    ]
    resultados['archivos_criticos'] = all(
        Path(a).exists() for a in archivos_criticos
    )

    # 6. Logs configurados
    resultados['directorio_logs'] = Path('logs').is_dir()

    # 7. Rendimiento básico (<200ms por predicción)
    try:
        tiempos = []
        for _ in range(5):
            t0 = time.perf_counter()
            requests.post(f'{base_url}/predict', json=vino_test, timeout=10)
            tiempos.append((time.perf_counter() - t0) * 1000)
        p95 = sorted(tiempos)[4]
        resultados['rendimiento_ok'] = p95 < 200
        resultados['latencia_p95_ms'] = round(p95, 1)
    except Exception:
        resultados['rendimiento_ok'] = False

    return resultados

# Lanzar servidor y ejecutar checklist
import subprocess, time
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.main:app', '--port', '8003', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)

if proc.poll() is None:
    resultados = checklist_produccion('http://localhost:8003')

    print("CHECKLIST DE PRODUCCIÓN")
    print('=' * 40)
    aprobados = 0
    total = 0
    for check, valor in resultados.items():
        if check == 'latencia_p95_ms':
            print(f"  Latencia P95: {valor}ms")
            continue
        total += 1
        estado = '✓' if valor else '✗'
        if valor:
            aprobados += 1
        print(f"  {estado} {check.replace('_', ' ').capitalize()}")

    print(f"\nResultado: {aprobados}/{total} comprobaciones superadas")
    if aprobados == total:
        print("✓ Sistema listo para producción")
    else:
        print("✗ Revisar las comprobaciones fallidas antes de desplegar")

    proc.terminate(); proc.wait()
else:
    print("Error al arrancar el servidor")

---
## Práctica

### Ejercicio 1 — Variables de entorno

**Enunciado:** Refactoriza `api/main.py` para leer la configuración desde variables de entorno:

```python
import os
MODEL_PATH  = os.getenv('MODEL_PATH', 'models/rf_vinos_v1.joblib')
LOG_LEVEL   = os.getenv('LOG_LEVEL', 'warning')
PORT        = int(os.getenv('PORT', '8000'))
UMBRAL      = float(os.getenv('UMBRAL_CALIDAD', '0.5'))
```

Luego demuestra que puedes cambiar el modelo sin tocar el código:
```bash
MODEL_PATH=models/dl_clf_v1.keras uvicorn api.main:app
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

Añade al inicio de `api/main.py`:

```python
import os
from pydantic_settings import BaseSettings  # pip install pydantic-settings

class Settings(BaseSettings):
    model_path:       str   = 'models/rf_vinos_v1.joblib'
    log_level:        str   = 'warning'
    port:             int   = 8000
    umbral_calidad:   float = 0.5
    workers:          int   = 2

    class Config:
        env_file = '.env'  # carga desde .env si existe

settings = Settings()

# Usar settings en el código:
# settings.model_path → ruta al modelo
# settings.umbral_calidad → umbral de clasificación
```

Crear un `.env.example` (no el `.env` real, que va en .gitignore):
```bash
# .env.example
MODEL_PATH=models/rf_vinos_v1.joblib
LOG_LEVEL=warning
PORT=8000
UMBRAL_CALIDAD=0.5
WORKERS=2
```

Prueba sin editar código:
```bash
MODEL_PATH=models/dl_clf_v1.keras uvicorn api.main:app --port 8000
```
</details>

---
### Ejercicio 2 — Imagen Docker optimizada

**Enunciado:** El Dockerfile que generamos incluye TensorFlow, que pesa ~500MB. Crea una versión optimizada para servir SOLO el modelo Random Forest:

1. Crea `requirements_minimal.txt` sin tensorflow
2. Crea `Dockerfile.minimal` que use esta imagen
3. Compara el tamaño de las dos imágenes: `docker images`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import subprocess
from pathlib import Path

# requirements_minimal.txt — sin tensorflow
reqs_minimal = """
fastapi>=0.110
uvicorn[standard]>=0.27
gunicorn>=21
pydantic>=2.5
scikit-learn>=1.4
numpy>=1.26
pandas>=2.2
joblib>=1.3
""".strip()
Path('requirements_minimal.txt').write_text(reqs_minimal)

# Dockerfile.minimal
dockerfile_min = '''FROM python:3.11-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY requirements_minimal.txt .
RUN pip install --no-cache-dir -r requirements_minimal.txt
COPY api/ ./api/
COPY models/rf_vinos_v1.joblib ./models/
RUN mkdir -p data/db logs
EXPOSE 8000
CMD ["gunicorn", "api.main:app",
     "--workers", "2",
     "--worker-class", "uvicorn.workers.UvicornWorker",
     "--bind", "0.0.0.0:8000"]
'''
Path('Dockerfile.minimal').write_text(dockerfile_min)

# Construir
result = subprocess.run(
    ['docker', 'build', '-f', 'Dockerfile.minimal', '-t', 'wine-api:minimal', '.'],
    capture_output=True, text=True
)
if result.returncode == 0:
    # Comparar tamaños
    imgs = subprocess.run(['docker', 'images', 'wine-api'], capture_output=True, text=True)
    print(imgs.stdout)
else:
    print(result.stderr[-500:])
```
</details>

---
### Ejercicio 3 — Script de despliegue automático

**Enunciado:** Crea `scripts/deploy.sh` que automatice el despliegue completo:

1. Comprueba que Docker esté instalado
2. Construye la imagen nueva
3. Detiene la versión anterior si está corriendo
4. Arranca la nueva
5. Espera al health check
6. Si el health check falla, hace rollback a la imagen anterior

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```bash
#!/bin/bash
# scripts/deploy.sh — despliegue con rollback automático
set -e

IMAGE="wine-api"
CONTAINER="wine_api"
PORT="8000"
HEALTH_URL="http://localhost:${PORT}/health"

# 1. Comprobar Docker
if ! command -v docker &> /dev/null; then
    echo "ERROR: Docker no está instalado" && exit 1
fi

# 2. Guardar versión anterior
OLD_IMAGE=$(docker inspect ${CONTAINER} --format='{{.Config.Image}}' 2>/dev/null || echo "")
echo "Versión anterior: ${OLD_IMAGE:-'ninguna'}"

# 3. Construir nueva imagen
echo "Construyendo imagen..."
docker build -t ${IMAGE}:new . || { echo "ERROR en build"; exit 1; }

# 4. Parar y eliminar el contenedor anterior
docker stop ${CONTAINER} 2>/dev/null || true
docker rm   ${CONTAINER} 2>/dev/null || true

# 5. Arrancar nueva versión
echo "Arrancando nueva versión..."
docker run -d --name ${CONTAINER} -p ${PORT}:8000 \
    -v $(pwd)/data:/app/data \
    -v $(pwd)/models:/app/models \
    -v $(pwd)/logs:/app/logs \
    ${IMAGE}:new

# 6. Esperar health check
echo "Esperando health check..."
for i in $(seq 1 12); do
    sleep 5
    if curl -sf ${HEALTH_URL} > /dev/null 2>&1; then
        echo "✓ Despliegue exitoso en ${HEALTH_URL}"
        docker tag ${IMAGE}:new ${IMAGE}:latest
        exit 0
    fi
    echo "  Intento ${i}/12..."
done

# 7. Rollback
echo "ERROR: Health check fallido. Haciendo rollback..."
docker stop ${CONTAINER} && docker rm ${CONTAINER}
if [ -n "${OLD_IMAGE}" ]; then
    docker run -d --name ${CONTAINER} -p ${PORT}:8000 ${OLD_IMAGE}
    echo "Rollback completado a ${OLD_IMAGE}"
fi
exit 1
```
```bash
chmod +x scripts/deploy.sh && ./scripts/deploy.sh
```
</details>

---
## Cierre del Módulo 9

¡Enhorabuena! Has construido un **Sistema Inteligente de Predicción y Exposición de Modelos** completo:

```
Clase 1 — Datos:       Descarga Wine Quality UCI con integridad SHA256
Clase 2 — Limpieza:    Outliers, encoding, feature engineering
Clase 3 — SQL:         SQLite + SQLAlchemy + pandas ↔ SQL
Clase 4 — ML:          Random Forest + GridSearchCV + joblib
Clase 5 — DL:          TensorFlow/Keras clf + reg + callbacks
Clase 6 — API:         FastAPI + Pydantic + uvicorn
Clase 7 — Docs:        Docstrings + README + OpenAPI + Git
Clase 8 — Despliegue:  systemd + Docker + Docker Compose
```

### Próximos pasos como AI Engineer

- **Nube**: desplegar en AWS/GCP/Azure (EC2, Cloud Run, App Service)
- **MLOps**: MLflow para experimentos, DVC para versionado de datos
- **Monitorización**: Prometheus + Grafana para métricas en producción
- **CI/CD**: GitHub Actions para test + despliegue automático
- **Escala**: Kubernetes para múltiples réplicas y alta disponibilidad

---
## Resumen final

```bash
# Desarrollo local
uvicorn api.main:app --reload --port 8000

# Producción con gunicorn
gunicorn api.main:app --workers 2 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000

# Docker
docker build -t wine-api .
docker run -d -p 8000:8000 wine-api

# Docker Compose
docker compose up -d
docker compose ps
docker compose logs -f
docker compose down

# systemd (Linux, requiere root)
sudo systemctl start wine-api
sudo systemctl enable wine-api
sudo journalctl -u wine-api -f
```